In [2]:
print("""
@File         : creating_binary_variables_through_one-hot_encoding.ipynb
@Author(s)    : Stephen CUI
@LastEditor(s): Stephen CUI
@CreatedTime  : 2025-01-29 18:13:52
@Email        : cuixuanstephen@gmail.com
@Description  : 通过独热编码创建二进制变量
""")


@File         : creating_binary_variables_through_one-hot_encoding.ipynb
@Author(s)    : Stephen CUI
@LastEditor(s): Stephen CUI
@CreatedTime  : 2025-01-29 18:13:52
@Email        : cuixuanstephen@gmail.com
@Description  : 通过独热编码创建二进制变量



In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [4]:
data = pd.read_csv('credit_approval_uci.csv')

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    data.drop(labels=["target"], axis=1),
    data["target"],
    test_size=0.3,
    random_state=0,
)

In [6]:
X_train['A4'].unique()

array(['u', 'y', 'Missing', 'l'], dtype=object)

In [7]:
dummies = pd.get_dummies(X_train['A4'], drop_first=True)

dummies.head()

,l,u,y
596,False,True,False
303,False,True,False
204,False,False,True
351,False,False,True
118,False,True,False


With pandas’ `get_dummies()`, we can either ignore or encode missing data through the `dummy_na` parameter. By setting `dummy_na=True`, missing data will be encoded in a new binary variable. To encode the variable into k dummies, use `drop_first=False` instead.

In [8]:
X_train_enc = pd.get_dummies(X_train, drop_first=True)
X_test_enc = pd.get_dummies(X_test, drop_first=True)

In [9]:
X_train_enc.head()

,A2,A3,A8,A11,A14,A15,A1_a,A1_b,A4_l,A4_u,...,A7_j,A7_n,A7_o,A7_v,A7_z,A9_t,A10_t,A12_t,A13_p,A13_s
596,46.08,3.000,2.375,8,396.0,4159,True,False,False,True,...,False,False,False,True,False,True,True,True,False,False
303,15.92,2.875,0.085,0,120.0,0,True,False,False,True,...,False,False,False,True,False,False,False,False,False,False
204,36.33,2.125,0.085,1,50.0,1187,False,True,False,False,...,False,False,False,True,False,True,True,False,False,False
351,22.17,0.585,0.000,0,100.0,0,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
118,57.83,7.040,14.000,6,360.0,1332,False,True,False,True,...,False,False,False,True,False,True,True,True,False,False


pandas’ `get_dummies()` will encode all variables of the object, string, or category type by default. To encode a subset of the variables, pass the variable names in a list to the `columns` parameter.

pandas 的 `get_dummies()` 将为 DataFrame 中看到的每个类别创建一个二进制变量。因此，如果训练集中的类别多于测试集中的类别，则 `get_dummies()` 将返回转换后的训练集中比转换后的测试集中更多的列，反之亦然。为了避免这种情况，最好使用 scikit‑learn 或 feature‑engine 进行独热编码。

In [10]:
from sklearn.preprocessing import  OneHotEncoder
from sklearn.compose import ColumnTransformer

In [11]:
cat_vars = X_train.select_dtypes(include='O').columns.to_list()
cat_vars

['A1', 'A4', 'A5', 'A6', 'A7', 'A9', 'A10', 'A12', 'A13']

In [12]:
encoder = OneHotEncoder(drop='first', sparse_output=False)

> 要将变量编码为 k 个虚拟变量，请将 `drop` 参数设置为 `None`。要仅将二进制变量编码为 k‑1，请将 `drop` 参数设置为 `if_binary`。后者很有用，因为将二进制变量编码为 k 个虚拟变量是多余的。

In [13]:
ct = ColumnTransformer(
    [('encoder', encoder, cat_vars)],
    remainder='passthrough',
    force_int_remainder_cols=False
).set_output(transform='pandas')

In [14]:
ct.fit(X_train)

ColumnTransformer(force_int_remainder_cols=False, remainder='passthrough',
                  transformers=[('encoder',
                                 OneHotEncoder(drop='first',
                                               sparse_output=False),
                                 ['A1', 'A4', 'A5', 'A6', 'A7', 'A9', 'A10',
                                  'A12', 'A13'])])

In [16]:
ct.named_transformers_['encoder'].categories_

[array(['Missing', 'a', 'b'], dtype=object),
 array(['Missing', 'l', 'u', 'y'], dtype=object),
 array(['Missing', 'g', 'gg', 'p'], dtype=object),
 array(['Missing', 'aa', 'c', 'cc', 'd', 'e', 'ff', 'i', 'j', 'k', 'm',
        'q', 'r', 'w', 'x'], dtype=object),
 array(['Missing', 'bb', 'dd', 'ff', 'h', 'j', 'n', 'o', 'v', 'z'],
       dtype=object),
 array(['f', 't'], dtype=object),
 array(['f', 't'], dtype=object),
 array(['f', 't'], dtype=object),
 array(['g', 'p', 's'], dtype=object)]

> scikit‑learn 的 `OneHotEncoder()` 只会对从训练集中学习到的类别进行编码。如果测试集中有新的类别，我们可以通过将 `handle_unknown` 参数设置为 `ignore`、`error` 或 `infrequent_if_exists` 来指示编码器忽略它们、返回错误或用不常见的类别替换它们。

In [17]:
X_train_enc = ct.transform(X_train)
X_test_enc = ct.transform(X_test)

In [18]:
ct.get_feature_names_out()

array(['encoder__A1_a', 'encoder__A1_b', 'encoder__A4_l', 'encoder__A4_u',
       'encoder__A4_y', 'encoder__A5_g', 'encoder__A5_gg',
       'encoder__A5_p', 'encoder__A6_aa', 'encoder__A6_c',
       'encoder__A6_cc', 'encoder__A6_d', 'encoder__A6_e',
       'encoder__A6_ff', 'encoder__A6_i', 'encoder__A6_j',
       'encoder__A6_k', 'encoder__A6_m', 'encoder__A6_q', 'encoder__A6_r',
       'encoder__A6_w', 'encoder__A6_x', 'encoder__A7_bb',
       'encoder__A7_dd', 'encoder__A7_ff', 'encoder__A7_h',
       'encoder__A7_j', 'encoder__A7_n', 'encoder__A7_o', 'encoder__A7_v',
       'encoder__A7_z', 'encoder__A9_t', 'encoder__A10_t',
       'encoder__A12_t', 'encoder__A13_p', 'encoder__A13_s',
       'remainder__A2', 'remainder__A3', 'remainder__A8',
       'remainder__A11', 'remainder__A14', 'remainder__A15'], dtype=object)

In [20]:
from feature_engine.encoding import OneHotEncoder

ohe_enc = OneHotEncoder(drop_last=True)

feature‑engine 的 `OneHotEncoder()` 默认对所有分类变量进行编码。要对变量子集进行编码，请将变量名称传递到列表中：`OneHotEncoder(variables=["A1", "A4"])`。要对数值变量进行编码，请将 `ignore_format` 参数设置为 `True` 或将变量转换为对象。

In [21]:
ohe_enc.fit(X_train)

OneHotEncoder(drop_last=True)

> 要将二元变量编码为 k‑1，并将其他分类变量编码为 k 个虚拟变量，请将 `drop_last_binary` 参数设置为 True。

In [22]:
ohe_enc.variables_

['A1', 'A4', 'A5', 'A6', 'A7', 'A9', 'A10', 'A12', 'A13']

In [23]:
ohe_enc.encoder_dict_

{'A1': ['a', 'b'],
 'A4': ['u', 'y', 'Missing'],
 'A5': ['g', 'p', 'Missing'],
 'A6': ['c',
  'q',
  'w',
  'ff',
  'm',
  'i',
  'e',
  'cc',
  'x',
  'd',
  'k',
  'j',
  'Missing',
  'aa'],
 'A7': ['v', 'ff', 'h', 'dd', 'z', 'bb', 'j', 'Missing', 'n'],
 'A9': ['t'],
 'A10': ['t'],
 'A12': ['t'],
 'A13': ['g', 's']}

In [24]:
X_train_enc = ohe_enc.transform(X_train)
X_test_enc = ohe_enc.transform(X_test)

In [25]:
X_train_enc.head()

,A2,A3,A8,A11,A14,A15,A1_a,A1_b,A4_u,A4_y,...,A7_z,A7_bb,A7_j,A7_Missing,A7_n,A9_t,A10_t,A12_t,A13_g,A13_s
596,46.08,3.000,2.375,8,396.0,4159,1,0,1,0,...,0,0,0,0,0,1,1,1,1,0
303,15.92,2.875,0.085,0,120.0,0,1,0,1,0,...,0,0,0,0,0,0,0,0,1,0
204,36.33,2.125,0.085,1,50.0,1187,0,1,0,1,...,0,0,0,0,0,1,1,0,1,0
351,22.17,0.585,0.000,0,100.0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,1,0
118,57.83,7.040,14.000,6,360.0,1332,0,1,1,0,...,0,0,0,0,0,1,1,1,1,0


> 独热编码适用于线性模型。它还可以扩展特征空间。如果数据集包含许多分类变量或高基数变量，则可以通过仅对最常见类别进行编码来限制二进制变量的数量。可以使用 scikit‑learn 和 feature‑engine 自动执行此操作，正如我们在执行频繁类别的独热编码配方中所述。